In [11]:
import numpy as np 

In [12]:
# Activation Function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
# Derivative of Sigmoid Function
def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def softmax(x):
    exp_x = np.exp(x - np.max(x))  
    return exp_x / np.sum(exp_x, axis=0)

In [13]:
class MLP:
    def __init__(self,input_sz,hidden_sz,output_sz):
        self.input_sz = input_sz
        self.hidden_sz = hidden_sz
        self.output_sz = output_sz
        
        # Initializing weights and biases
        self.W1 = np.random.rand(self.input_sz, self.hidden_sz) * 0.01
        self.b1 = np.zeros((1, self.hidden_sz))
        self.W2 = np.random.rand(self.hidden_sz, self.output_sz) * 0.01
        self.b2 = np.zeros((1, self.output_sz))
    
    def forward(self,X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        return self.a2
    def backward(self,X,y_true):
        m = X.shape[0]
        # Output layer errors
        dz2 = self.a2 - y_true
        dW2 = np.dot(self.a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m
        
        # Hidden layer errors
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * sigmoid_derivative(self.z1)
        dW1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m
        
        # Update weights and biases
        self.W2 -= 0.01 * dW2
        self.b2 -= 0.01 * db2
        self.W1 -= 0.01 * dW1
        self.b1 -= 0.01 * db1

In [14]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time


In [15]:
class torch_MLP(nn.Module):
    def __init__(self):
        super(torch_MLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
class torch_CNN(nn.Module):
    def __init__(self):
        super(torch_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [16]:

from torchvision import datasets, transforms

# normalization of dataset
transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,)),])

# downloading Traing data
train_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

# test dataset
test_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=True)


In [17]:

input_size = 28 * 28
hidden_size = 128
output_size = 10
numpy_mlp = MLP(input_size, hidden_size, output_size)

# Traing Loop for MLP
epochs = 5
for epoch in range(epochs):
    for images, labels in train_loader:
        # flatening images
        images = images.numpy().reshape(images.shape[0], -1)
        
        # OHL for loss cal
        y_true = np.zeros((labels.shape[0], 10))
        y_true[np.arange(labels.shape[0]), labels.numpy()] = 1
        
        # FP
        numpy_mlp.forward(images)
        
        # BP
        numpy_mlp.backward(images, y_true)
    print(f"Epoch {epoch+1}/{epochs}")


Epoch 1/5
Epoch 2/5
Epoch 3/5
Epoch 4/5
Epoch 5/5


In [18]:

# Evaluation
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        # Flatting images
        images = images.numpy().reshape(images.shape[0], -1)
        labels = labels.numpy()
        
        # model prediction
        outputs = numpy_mlp.forward(images)
        
        # class with highest prob
        predicted = np.argmax(outputs, axis=1)
        
        # Comparing with true label
        total += labels.shape[0]
        correct += (predicted == labels).sum()

accuracy = 100 * correct / total
print(f'Numpy_MLP: {accuracy:.2f} %')


Numpy_MLP: 14.11 %


In [19]:

# Define a simple network for y = x
class IdentityNetwork(nn.Module):
    def __init__(self):
        super(IdentityNetwork, self).__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

# Create the model and the optimizer
identity_model = IdentityNetwork()
optimizer = optim.SGD(identity_model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Create some training data
X_train = torch.randn(100, 1) * 10
y_train = X_train.clone()

# Train the network
for epoch in range(100):
    optimizer.zero_grad()
    output = identity_model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

# Print the learned weights
for name, param in identity_model.named_parameters():
    if param.requires_grad:
        print(name, param.data)


linear.weight tensor([[-1.7659e+08]])
linear.bias tensor([232196.7969])


In [20]:

# Define a network for y = x^2
class SquaringNetwork(nn.Module):
    def __init__(self):
        super(SquaringNetwork, self).__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Create the model and the optimizer
squaring_model = SquaringNetwork()
optimizer = optim.Adam(squaring_model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Create some training data
X_train = torch.randn(200, 1) * 10
y_train = X_train.pow(2)

# Train the network
for epoch in range(200):
    optimizer.zero_grad()
    output = squaring_model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/200], Loss: {loss.item():.4f}')


Epoch [20/200], Loss: 17700.7871
Epoch [40/200], Loss: 6740.3809
Epoch [60/200], Loss: 3709.4756
Epoch [80/200], Loss: 3492.7703
Epoch [100/200], Loss: 3293.1194
Epoch [120/200], Loss: 3121.9060
Epoch [140/200], Loss: 2959.5706
Epoch [160/200], Loss: 2800.5879
Epoch [180/200], Loss: 2643.7678
Epoch [200/200], Loss: 2488.2693
